### Loading the data from Part 1

In [1]:
import pickle
import pandas as pd
import networkx as nx

# disease-level ppi
with open("../data/ppi.pkl", "rb") as f:
    ppi = pickle.load(f)

# species full biogrid ppi
with open("../data/biogrid.pkl", "rb") as f:
    biogrid = pickle.load(f)

# human to other organism mapping
mapping = pd.read_csv("../data/homolog_mapping.csv")

# networks
with open("../data/networks.pkl", "rb") as f:
    networks = pickle.load(f)

# hubs
with open("../data/hubs.pkl", "rb") as f:
    hubs = pickle.load(f)

### Load disease modules and calculate gene connectivity within modules

In [2]:
from networkx.algorithms.community import louvain_communities

diseases = list(networks.keys())

RANDOM_SEED = 42
MIN_MODULE_SIZE = 5

def find_modules(G, min_size=MIN_MODULE_SIZE, seed=RANDOM_SEED):
    communities = louvain_communities(G, seed=seed)
    return [set(c) for c in communities if len(c) >= min_size]

modules = {d: find_modules(networks[d]["human"]) for d in diseases}

# Get gene level connectivity features for each disease module
rows = []
for d in diseases:
    G_h = networks[d]["human"]

    for module_id, module in enumerate(modules[d]):

        H = G_h.subgraph(module)
        module_degree = dict(H.degree())

        for gene in module:

            rows.append({
                "gene": gene,
                "disease": d,
                "module_id": module_id,

                # Gene degree within the module
                "module_degree": module_degree.get(gene, 0),

                # Degree of the gene in the full human PPI network
                "global_degree": G_h.degree(gene),

                "module_size": len(module)
            })

gene_df = pd.DataFrame(rows)

gene_df.head()

,gene,disease,module_id,module_degree,global_degree,module_size
0,ROR1,AD,0,1,1,194
1,FGFR3,AD,0,1,1,194
2,MSR1,AD,0,1,1,194
3,AXL,AD,0,2,3,194
4,RANBP1,AD,0,1,1,194


### Read in conservation scores from part 2 and merge with the gene-level connectivity data


In [3]:
scores_df = pd.read_csv("../data/conservation_scores.csv")

gene_df = gene_df.merge(
    scores_df[["disease", "module_id", "species", "node_conservation"]],
    on=["disease", "module_id"],
    how="left"
)

gene_df.head()

,gene,disease,module_id,module_degree,global_degree,module_size,species,node_conservation
0,ROR1,AD,0,1,1,194,mouse,0.887681
1,ROR1,AD,0,1,1,194,yeast,0.206522
2,ROR1,AD,0,1,1,194,fly,0.485507
3,FGFR3,AD,0,1,1,194,mouse,0.887681
4,FGFR3,AD,0,1,1,194,yeast,0.206522


### Download proteomes for all organisms

In [4]:
import requests

proteomes = {
    "human": "UP000005640",
    "mouse": "UP000000589",
    "fly": "UP000000803",
    "yeast": "UP000002311"
}

base_url = "https://rest.uniprot.org/uniprotkb/stream"

for name, proteome_id in proteomes.items():
    url = f"{base_url}?format=fasta&query=(proteome:{proteome_id})"
    
    print(f"Downloading {name}...")
    r = requests.get(url, stream=True)
    
    if r.status_code == 200:
        with open(f"../data/proteomes/{name}.fasta", "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
        print(f"Saved {name}.fasta")
    else:
        print(f"Failed for {name}: {r.status_code}")

Saved human.fasta
Saved mouse.fasta
Saved fly.fasta
Saved yeast.fasta


### Filter Human proteome to our genes of interest

In [5]:
from Bio import SeqIO
import re

human_fasta = "../data/proteomes/human.fasta"
output_fasta = "../data/proteomes/human_network.fasta"

# collect network genes
network_genes = set()
for d in networks:
    network_genes |= set(networks[d]["human"].nodes())

network_genes = {g.upper() for g in network_genes}

filtered_records = []

for record in SeqIO.parse(human_fasta, "fasta"):
    header = record.description

    # extract GN= field properly
    match = re.search(r"GN=([A-Za-z0-9_-]+)", header)
    if not match:
        continue

    gene = match.group(1).upper()

    if gene in network_genes:
        filtered_records.append(record)

SeqIO.write(filtered_records, output_fasta, "fasta")

print(f"Network genes: {len(network_genes)}")
print(f"Filtered proteins: {len(filtered_records)}")

Network genes: 6430
Filtered proteins: 33479


### Run Blast

blastp -query ./human_network.fasta \
       -db ./blast_db/human/human \
       -num_threads 10 \
       -max_target_seqs 1 \
       -out ./blast_results/human_vs_human.tsv \
       -outfmt "6 qseqid sseqid bitscore evalue"

blastp -query ./human_network.fasta \
       -db ./blast_db/mouse/mouse \
       -num_threads 10 \
       -max_target_seqs 1 \
       -out ./blast_results/human_vs_mouse.tsv \
       -outfmt "6 qseqid sseqid bitscore evalue"

blastp -query ./human_network.fasta \
       -db ./blast_db/yeast/yeast \
       -num_threads 10 \
       -max_target_seqs 1 \
       -out ./blast_results/human_vs_yeast.tsv \
       -outfmt "6 qseqid sseqid bitscore evalue"

blastp -query ./human_network.fasta \
       -db ./blast_db/fly/fly \
       -num_threads 10 \
       -max_target_seqs 1 \
       -out ./blast_results/human_vs_fly.tsv \
       -outfmt "6 qseqid sseqid bitscore evalue"

### Map UniProt accessions to Gene names

In [6]:
mapping = []

with open("../data/proteomes/human_network.fasta") as f:
    for line in f:
        if line.startswith(">"):
            header = line.strip()
            
            # extract UniProt accession
            acc = header.split("|")[1]
            
            # extract gene name (GN=...)
            match = re.search(r"GN=([A-Za-z0-9_-]+)", header)
            gene = match.group(1) if match else None
            
            if gene:
                mapping.append((acc, gene))

mapping_df = pd.DataFrame(mapping, columns=["uniprot", "gene"])
print(mapping_df.head())


      uniprot      gene
0  A0A087WVL8      FMR1
1  A0A087WXI3      FMR1
2  A0A087WY29      FMR1
3  A0A087WYG2  MAPK8IP3
4  A0A087X033      PTEN


### Calculate bit score ratios (BSR) for each gene/species pair

In [13]:
cols = ["qseqid", "sseqid", "bitscore", "evalue"]

files = {
    "self":  "../data/proteomes/blast_results/human_vs_human.tsv",
    "fly":   "../data/proteomes/blast_results/human_vs_fly.tsv",
    "mouse": "../data/proteomes/blast_results/human_vs_mouse.tsv",
    "yeast": "../data/proteomes/blast_results/human_vs_yeast.tsv"
}

dfs = {k: pd.read_csv(v, sep="\t", names=cols) for k, v in files.items()}


def get_acc(x):
    try:
        return x.split("|")[1]
    except:
        return x

for k in dfs:
    dfs[k]["uniprot"] = dfs[k]["qseqid"].apply(get_acc)

# Bit score of the reference sequence
self_scores = (
    dfs["self"]
    .groupby("uniprot", as_index=False)["bitscore"]
    .max()
    .rename(columns={"bitscore": "self_bitscore"})
)

self_scores = self_scores.merge(mapping_df, on="uniprot", how="left")

# Take max self bitscore for each gene (in case multiple isoforms map to a single gene)
self_scores = (
    self_scores
    .dropna(subset=["gene"])
    .groupby("gene", as_index=False)["self_bitscore"]
    .max()
    .drop_duplicates(subset=["gene"])
)


# Calculate bitscores for each species
species_scores = {}

for sp in ["fly", "mouse", "yeast"]:
    tmp = (
        dfs[sp][["uniprot", "bitscore"]]
        .rename(columns={"bitscore": f"bitscore_{sp}"})
        .merge(mapping_df, on="uniprot", how="left")
    )

    tmp = (
        tmp
        .dropna(subset=["gene"])
        .groupby("gene", as_index=False)[f"bitscore_{sp}"]
        .max()
        .drop_duplicates(subset=["gene"])
    )

    species_scores[sp] = tmp


bsr_df = gene_df.copy()

bsr_df = bsr_df.merge(
    self_scores[["gene", "self_bitscore"]],
    on="gene",
    how="left"
)

for sp in ["fly", "mouse", "yeast"]:
    tmp = species_scores[sp][["gene", f"bitscore_{sp}"]]
    bsr_df = bsr_df.merge(tmp, on="gene", how="left")


# Match BSR to the correct species
def compute_bsr(row):
    if pd.isna(row["self_bitscore"]):
        return None
    
    sp = row["species"]
    
    # Cap BSR at 1.0 to avoid cases where bitscore is higher than self bitscore due to isoforms or noise
    # Maximum on our dataset was 1.001 without capping
    if sp == "fly":
        return min(row["bitscore_fly"] / row["self_bitscore"], 1.0)
    elif sp == "mouse":
        return min(row["bitscore_mouse"] / row["self_bitscore"], 1.0)
    elif sp == "yeast":
        return min(row["bitscore_yeast"] / row["self_bitscore"], 1.0)
    else:
        return None

bsr_df["BSR"] = bsr_df.apply(compute_bsr, axis=1)

print(bsr_df[["gene", "species", "BSR"]].head())
print(bsr_df["BSR"].describe())

bsr_df = bsr_df.drop(columns=[
    "bitscore_fly",
    "bitscore_mouse",
    "bitscore_yeast",
    "self_bitscore"
], errors="ignore")

bsr_df.to_csv("../data/bitscores.csv", index=False)

    gene species       BSR
0   ROR1   mouse  0.972393
1   ROR1   yeast  0.045706
2   ROR1     fly  0.204499
3  FGFR3   mouse  0.910951
4  FGFR3   yeast  0.063177
count    31242.000000
mean         0.424160
std          0.356067
min          0.001789
25%          0.073233
50%          0.320850
75%          0.793469
max          1.000000
Name: BSR, dtype: float64


### Node Connectivity Scores

In [16]:
bsr_df["mod_deg_norm"] = bsr_df["module_degree"] / (bsr_df["module_size"] - 1)

bsr_df["mod_z_score"] = (
    bsr_df.groupby("module_id")["mod_deg_norm"]
      .transform(lambda x: (x - x.mean()) / x.std())
)
bsr_df.to_csv("../data/bitscores.csv", index=False)